### Imports

In [12]:
import os
import random
import pandas as pd
import numpy as np
import torch
import logging
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import json
from sklearn.model_selection import KFold
from itertools import product
from sklearn.metrics import r2_score


if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
    print("Apple GPU")
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU / AMD ROCm GPU
    print("Nvidia/AMD GPU")
else:
    device = torch.device("cpu")
    print("CPU")


Apple GPU


## ML

### Functions for data management

In [5]:
def split_csvfiles(datafolder, random_seed, training_prop, validation_prop):
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(f)
    
    random.seed(random_seed)
    random.shuffle(csv_files)

    train_n = int(len(csv_files) * training_prop)
    val_n = int(len(csv_files) * validation_prop)
    test_n = len(csv_files) - train_n - val_n

    # Split
    if validation_prop == 0:
        train_files = csv_files[:train_n]
        test_files = csv_files[train_n:]

        return train_files, test_files
    
    else:
        train_files = csv_files[:train_n]
        val_files = csv_files[train_n: train_n + val_n]
        test_files = csv_files[train_n + val_n:]

        return train_files, val_files, test_files
    


def load(files, data_dir):
    dataframes = []

    for f in files:
        path = os.path.join(data_dir, f)   
        df = pd.read_csv(path)

        # Get rid of trailing whitespace
        df.columns = df.columns.str.strip()           
        dataframes.append(df)             

    combined = pd.concat(dataframes, ignore_index=True)  # combine all

    return combined


def input_target_split(dataframe):
    input_col_names = []
    target_col_names = []

    joints = ["head", "left_shoulder", "left_elbow", "right_shoulder", "right_elbow", "left_hand", "right_hand",
              "left_hip", "right_hip", "left_knee", "right_knee", "left_foot", "right_foot"]
    
    for joint in joints:
        input_col_names += [f"{joint}_x", f"{joint}_y"]
        target_col_names += [f"{joint}_z"]


    input_data = dataframe[input_col_names].copy()
    target_data = dataframe[target_col_names].copy()
    
    return input_data, target_data

### Load data


In [ ]:
datafolder = "../../MainProject/Assignment9/data/kinect_good_preprocessed"
random_seed = 42

# Alt 1: 80/10/10 split
train_files, val_files, test_files = split_csvfiles(datafolder, random_seed, 0.8, 0.1)
val_data = load(val_files, datafolder)

# Alt 2: 90/10 since we don't need validation data if we run k-fold cv
# train_files, test_files = split_csvfiles(datafolder, random_seed, 0.9, 0)

train_data = load(train_files, datafolder)
test_data = load(test_files, datafolder)

print(train_data.columns.tolist())


['FrameNo', 'head_x', 'head_y', 'head_z', 'left_shoulder_x', 'left_shoulder_y', 'left_shoulder_z', 'left_elbow_x', 'left_elbow_y', 'left_elbow_z', 'right_shoulder_x', 'right_shoulder_y', 'right_shoulder_z', 'right_elbow_x', 'right_elbow_y', 'right_elbow_z', 'left_hand_x', 'left_hand_y', 'left_hand_z', 'right_hand_x', 'right_hand_y', 'right_hand_z', 'left_hip_x', 'left_hip_y', 'left_hip_z', 'right_hip_x', 'right_hip_y', 'right_hip_z', 'left_knee_x', 'left_knee_y', 'left_knee_z', 'right_knee_x', 'right_knee_y', 'right_knee_z', 'left_foot_x', 'left_foot_y', 'left_foot_z', 'right_foot_x', 'right_foot_y', 'right_foot_z']


### Input/target creation

In [11]:

x_train, y_train = input_target_split(train_data)
x_val, y_val = input_target_split(val_data)
x_test, y_test = input_target_split(test_data)


print("Train:", x_train.shape, y_train.shape)
print("Val:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)


x_train = torch.tensor(x_train.values, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train.values, dtype=torch.float32).to(device)
x_val = torch.tensor(x_val.values, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val.values, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test.values, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test.values, dtype=torch.float32).to(device)




Train: (19653, 26) (19653, 13)
Val: (2105, 26) (2105, 13)
Test: (2247, 26) (2247, 13)


### Functions for learning

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
        nn.init.zeros_(m.bias)

class ZPredictor(nn.Module):
    def __init__(self, hidden_layers: list, activation="relu", dropout=0.0):
        super().__init__()
        
        layers = []
        input_size = 26  # 13 joints x 2 (x, y)

        activations = {"relu": nn.ReLU(),
                       "tanh": nn.Tanh(),
                       "gelu": nn.GELU(),
                       "leaky_relu": nn.LeakyReLU()
                       }
        
        act = activations[activation]
        prev_size = input_size

        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(act)

            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, 13))  # 13 joints z output
        
        self.network = nn.Sequential(*layers)

        self.network.apply(init_weights)
    
    def forward(self, x):
        return self.network(x)


def build_model(config):
    return ZPredictor(
        hidden_layers=config["hidden_layers"],
        activation=config["activation"],
        dropout=config["dropout"],
    ).to(device)

def compute_metrics(y_true, y_pred):
    mse = torch.mean((y_pred - y_true) ** 2).item()
    mae = torch.mean(torch.abs(y_pred - y_true)).item()
    bias = torch.mean(y_pred - y_true).item()

    y_true_np = y_true.detach().cpu().numpy()
    y_pred_np = y_pred.detach().cpu().numpy()
    r2 = r2_score(y_true_np, y_pred_np)

    return {"mse": mse, "mae": mae, "r2": r2, "bias": bias}

def evaluate_model(model, x_data, y_data, loss_fn):
    model.eval()
    with torch.no_grad():
        predictions = model(x_data)
        loss = loss_fn(predictions, y_data).item()
        metrics = compute_metrics(y_data, predictions)
    return loss, metrics, predictions


def train_one_model(model, config, x_train, y_train, x_val, y_val, verbose=False):
    optimizer_name = config["optimizer"]
    lr = config["learning_rate"]

    if optimizer_name == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == "rmsprop":
        optimizer = optim.RMSprop(model.parameters(), lr=lr)
    elif optimizer_name == "sgd":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    loss_fn = nn.MSELoss()
    epochs = config["epochs"]
    patience = config["patience"]

    best_val_loss = float("inf")
    best_state = None
    history = []
    epochs_no_improve = 0

    for epoch in range(epochs):
        # Train
        model.train()
        optimizer.zero_grad()
        train_predictions = model(x_train)
        train_loss = loss_fn(train_predictions, y_train)
        train_loss.backward()
        optimizer.step()

        # Evaluate
        val_loss, val_metrics, _ = evaluate_model(model, x_val, y_val, loss_fn)
        train_metrics = compute_metrics(y_train, train_predictions)

        row = {
            "epoch": epoch + 1,
            "train_loss": float(train_loss.item()),
            "val_loss": float(val_loss),
            "train_mae": train_metrics["mae"],
            "val_mae": val_metrics["mae"],
            "train_r2": train_metrics["r2"],
            "val_r2": val_metrics["r2"],
        }
        history.append(row)


        # Early stoping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    final_val_loss, final_val_metrics, _ = evaluate_model(model, x_val, y_val, loss_fn)
    return {
        "model": model,
        "best_state": best_state,
        "history": history,
        "val_loss": final_val_loss,
        "val_metrics": final_val_metrics,
    }


def cross_validation(config, X, Y, k):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_scores = []
    fold_models = []

    with mlflow.start_run():

        mlflow.log_params(config)
        for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
            x_train = X[train_idx]
            y_train = Y[train_idx]
            x_val = X[val_idx]
            y_val = Y[val_idx]

            model = build_model(config)
            model.apply(init_weights)

            val_loss, best_state = train_one_model(model, config, x_train, y_train, x_val, y_val, loss_fn = nn.MSELoss())

            fold_scores.append(val_loss)
            fold_models.append(best_state)

            mlflow.log_metric(f"fold_{fold}_val_loss ", val_loss)
        
        avg_loss = sum(fold_scores) / len(fold_scores)

        mlflow.log_metric("cv_mean_val_loss", avg_loss)
    
    return {"cv_mean_loss": avg_loss,
            "fold_scores": fold_scores,
            "fold_models": fold_models}

### Function for saving model

In [ ]:
# Base paths
base_model_dir = "assignment9_models"
candidates_dir = os.path.join(base_model_dir, "candidates")
champion_dir = os.path.join(base_model_dir, "champion")
metadata_dir = os.path.join(base_model_dir, "metadata")

# Create folders if they don't exist
# os.makedirs(candidates_dir, exist_ok=True)
# os.makedirs(champion_dir, exist_ok=True)
# os.makedirs(metadata_dir, exist_ok=True)




def save_candidate_model(model, model_name):
    path = os.path.join(candidates_dir, f"{model_name}.pt")
    torch.save(model.state_dict(), path)
    print(f"Saved candidate model: {path}")
    return path


def load_champion_info():
    path = os.path.join(metadata_dir, "champion_info.json")

    if not os.path.exists(path):
        return None

    try:
        with open(path, "r") as f:
            return json.load(f)
    except:
        return None  



def save_champion_model(model, model_name, mse, mae, hyperparameters):
    model_path = os.path.join(champion_dir, "champion_model.pt")
    info_path = os.path.join(metadata_dir, "champion_info.json")

    # Save model weights
    torch.save(model.state_dict(), model_path)

    # Save metadata
    info = {
        "model_name": model_name,
        "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "mse": float(mse),
        "mae": float(mae),
        "hyperparameters": hyperparameters
    }

    with open(info_path, "w") as f:
        json.dump(info, f, indent=2)

    print("New champion model saved!")



def update_champion(model, model_name, mse, mae, hyperparameters):
    current = load_champion_info()

    if current is None:
        print("No champion found --> saving first model")
        save_champion_model(model, model_name, mse, mae, hyperparameters)

    elif mse < current["mse"]:
        print(f"New model is better (MSE {mse} < {current['mse']})")
        save_champion_model(model, model_name, mse, mae, hyperparameters)

    else:
        print(f"Model NOT better (MSE {mse} ≥ {current['mse']})")